## Set IBCs
Import libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
sys.path.append(os.path.abspath('./functions'))

from thermos import *
from writing import *

Define inputs

In [ ]:
models = ['surrogate', 'thermolib']
schemes = ['strang']
nb_nodes = 1401
species = ['O', 'O2']
freestream = {
    'p': 1000.,
    'u': 3000.,
    'T': 500.,
    'X': [0.05, 0.95]
}
cols = ['x','X_O','X_O2','u','Th','Tv','rho','p','nd','H','Mf']
path_to_case = "../"

In [ ]:
path_to_case = os.path.abspath(path_to_case)

Read PLATO shock solution

In [ ]:
data = np.loadtxt(path_to_case + '/ode/mt/output_shock/shock.dat')
data = pd.DataFrame(data, columns=cols)

Initialize thermodynamics and extract primitive variables from PLATO solution

In [ ]:
sp_th, mix = init_thermos()

In [ ]:
prim_sol = extract_prim_var(data, sp_th, mix, species)

Compute freestream primitive variables

In [ ]:
# Get temperatures and molar fractions
T = np.array([freestream['T']]*2).reshape(1,2)
X = np.array(freestream['X']).reshape(1,2)
# Update thermodynamics
for s in sp_th.values():
    s.update(T)
# Update mixture
mix.update(X, var='X')
# Compute needed physical variables
u = np.array(freestream['u']).reshape(1,1)
rho = mix._rho(freestream['p'], T)
Y = np.hstack([sp_th[k].specie.Y.numpy() for k in species])
# Concatenate
prim_freestream = tf.concat([Y*rho, u, T], 1).numpy()

Write IBCs to file

In [ ]:
inlet_prim = np.array2string(
    prim_freestream.squeeze(),
    formatter={'float_kind': lambda x: "%.10e" % x}
).replace('\n', '')

init_sol_prim = np.array2string(
    np.hstack([prim_freestream, np.take(prim_sol, [-1], axis=0)]).squeeze(),
    formatter={'float_kind': lambda x: "%.10e" % x}
).replace('\n', '')

for model in models:
    for scheme in schemes:
        file = path_to_case + f'/pde/{model}/n{nb_nodes}/{scheme}/input/inputfile.sh'
        if os.path.exists(file):
            replace(
                file=os.path.abspath(file),
                patterns=(
                    "INLET.PRIM",
                    "INIT_SOL_PRIM"
                ),
                replaces=(
                    f"INLET.PRIM = {inlet_prim[1:-1]}\n",
                    f"INIT_SOL_PRIM = {init_sol_prim[1:-1]}\n"
                )
            )